In [7]:
from quant.data.repository import QuantRepository
from quant.data.db import DuckDBManager
from quant.config import load_config
import datetime

config = load_config()
raw_dir = config.paths.raw_dir
process_dir = config.paths.processed_dir
database_path = config.paths.database_path
dbManager = DuckDBManager(database_path, process_dir)
dbManager.initialize()
with dbManager.session() as conn:
    qr = QuantRepository(conn)
    td = qr.get_trade_calendar(datetime.date(2026,1,1),datetime.date(2026,6,20))
    print(td)


2026-06-30 16:13:36.181 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_daily_ohlcv
2026-06-30 16:13:36.194 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_adj_factor
2026-06-30 16:13:36.226 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_daily_basic
2026-06-30 16:13:36.227 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 index_daily
2026-06-30 16:13:36.227 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 fundamental
2026-06-30 16:13:36.228 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 factors
2026-06-30 16:13:36.238 | INFO     | quant.data.db:_create_derived_views:184 - 已注册 DuckDB 视图 v_daily_hfq
2026-06-30 16:13:36.243 | INFO     | quant.data.db:_create_derived_views:229 - 已注册 DuckDB 视图 v_daily_qfq_latest
2026-06-30 16:13:36.247 | INFO     | quant.data.db:_create_derived_views:254 - 已注册 DuckDB 兼容视图 v_daily_adj


[{'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 1), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 2), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 3), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 4), 'is_open': False, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 5), 'is_open': True, 'pretrade_date': datetime.date(2025, 12, 31)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 6), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 5)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 7), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 6)}, {'exchange': 'SSE', 'cal_date': datetime.date(2026, 1, 8), 'is_open': True, 'pretrade_date': datetime.date(2026, 1, 7)}, {'exchange': 'SSE

In [8]:
from quant.data.db import DuckDBManager
from quant.config import load_config
from datetime import date
import pandas as pd

def get_qfq_ohlcv_df(
    ts_code: str,
    start: date,
    end: date,
    *,
    as_of_date: date | None = None,
) -> pd.DataFrame:
    """查询单只股票前复权 OHLCV。
    前复权公式:
    qfq_price = raw_price * cumulative_factor / asof_cumulative_factor
    """
    config = load_config()
    manager = DuckDBManager(config.paths.database_path, config.paths.processed_dir)
    manager.initialize()
    effective_as_of = as_of_date or end 
    with manager.session() as conn:
        asof_row = conn.execute(
            """
            SELECT cumulative_factor
            FROM v_adj_factor
            WHERE ts_code = ?
                AND trade_date <= ?
            ORDER BY trade_date DESC
            LIMIT 1
            """,
            [ts_code, effective_as_of],
        ).fetchone()
        if asof_row is None:
            raise ValueError(f"缺少 as_of 复权因子: {ts_code}, {effective_as_of}")
        asof_factor = float(asof_row[0])
        df = conn.execute(
            """
            SELECT
                o.ts_code,
                o.trade_date,
                o.open,
                o.high,
                o.low,
                o.close,
                a.cumulative_factor,
                o.volume,
                o.amount,
                o.is_suspended,
                o.is_st,
                o.limit_status
            FROM v_daily_ohlcv o
            LEFT JOIN v_adj_factor a
                ON o.ts_code = a.ts_code
                AND o.trade_date = a.trade_date
            WHERE o.ts_code = ?
                AND o.trade_date BETWEEN ? AND ?
                AND o.trade_date <= ?
            ORDER BY o.trade_date
            """,
            [ts_code, start, end, effective_as_of],
        ).fetchdf()
    if df.empty:
        return df
    factor_ratio = df["cumulative_factor"] / asof_factor
    df["open"] = df["open"] * factor_ratio
    df["high"] = df["high"] * factor_ratio
    df["low"] = df["low"] * factor_ratio
    df["close"] = df["close"] * factor_ratio
    return df.drop(columns=["cumulative_factor"])

df = get_qfq_ohlcv_df(ts_code='000001.SZ',start=date(2026,1,1),end=date(2026,6,26))
print(df['open'])

2026-06-30 16:13:36.334 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_daily_ohlcv
2026-06-30 16:13:36.337 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_adj_factor
2026-06-30 16:13:36.352 | INFO     | quant.data.db:_register_parquet_views:151 - 已注册 DuckDB 视图 v_daily_basic
2026-06-30 16:13:36.353 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 index_daily
2026-06-30 16:13:36.353 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 fundamental
2026-06-30 16:13:36.354 | DEBUG    | quant.data.db:_register_parquet_views:138 - 跳过缺失的 Parquet 数据集 factors
2026-06-30 16:13:36.360 | INFO     | quant.data.db:_create_derived_views:184 - 已注册 DuckDB 视图 v_daily_hfq
2026-06-30 16:13:36.365 | INFO     | quant.data.db:_create_derived_views:229 - 已注册 DuckDB 视图 v_daily_qfq_latest
2026-06-30 16:13:36.369 | INFO     | quant.data.db:_create_derived_views:254 - 已注册 DuckDB 兼容视图 v_daily_adj


0      11.056175
1      11.133626
2      11.288529
3      11.269166
4      11.162670
         ...    
109    10.520000
110    10.650000
111    10.680000
112    10.470000
113    10.420000
Name: open, Length: 114, dtype: float64


In [9]:
import numpy as np
# 1.1 创建时间序列数据
dates = pd.date_range(start="2023-01-01", periods=15, freq="D")
df = pd.DataFrame(
    {
        "close": [
            100,
            101,
            102,
            99,
            98,
            103,
            105,
            104,
            106,
            108,
            np.nan,
            110,
            112,
            111,
            113,
        ],  # 包含一个 NaN
        "volume": np.random.randint(1000, 2000, 15),
    },
    index=dates,
)

print("原始数据 (Head):")
print(df.head(3))

原始数据 (Head):
            close  volume
2023-01-01  100.0    1438
2023-01-02  101.0    1970
2023-01-03  102.0    1685


In [10]:
# 1.2 缺失值处理
# ffill (forward fill): 用前一个有效值填充
df_filled = df.ffill()
print("\n缺失值处理 (ffill):")
print(df_filled.loc["2023-01-10":"2023-01-12"])  # type: ignore


缺失值处理 (ffill):
            close  volume
2023-01-10  108.0    1400
2023-01-11  108.0    1160
2023-01-12  110.0    1778


In [12]:
# 1.3 滚动窗口 (Rolling Window)
# 计算 5日均线
df["ma5"] = df["close"].rolling(window=5).mean()
print("\n滚动窗口 (MA5):")
print(df.tail(3))
print(df.head(5))


滚动窗口 (MA5):
            close  volume  ma5
2023-01-13  112.0    1164  NaN
2023-01-14  111.0    1995  NaN
2023-01-15  113.0    1926  NaN
            close  volume    ma5
2023-01-01  100.0    1438    NaN
2023-01-02  101.0    1970    NaN
2023-01-03  102.0    1685    NaN
2023-01-04   99.0    1631    NaN
2023-01-05   98.0    1454  100.0


In [13]:
# 1.4 重采样 (Resample)
# 将日线转为 5日线 (周线近似)
df_5d = df.resample("5D").agg({"close": "last", "volume": "sum"})
print("\n重采样 (5日线):")
print(df_5d)



重采样 (5日线):
            close  volume
2023-01-01   98.0    8178
2023-01-06  108.0    8014
2023-01-11  113.0    8023
